# AI-Driven Portfolio Optimizer Walkthrough

This notebook demonstrates an end-to-end workflow for an **AI-driven portfolio optimization** project:

1. Download historical stock prices with `yfinance`
2. Engineer technical indicators
3. Cluster stocks into risk groups with **K-Means**
4. Predict next-day returns with **Random Forest**
5. Optimize allocations with **PyPortfolioOpt**
6. Backtest strategies over a 5-year period
7. Visualize results with **Plotly**


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DEFAULT_TICKERS, RISK_LEVEL_MAP, TECHNICAL_FEATURES
from src.data_loader import download_price_data, compute_returns
from src.features import build_technical_features
from src.ml_models import cluster_stocks, prepare_ml_dataset, train_random_forest_models, predict_next_day_scores
from src.optimizer import optimize_portfolio
from src.backtest import run_backtest, summarize_performance
from src.visuals import plot_price_history, plot_cluster_map, plot_cumulative_returns, plot_feature_importance, plot_weights

print('Project modules imported successfully.')

c:\Users\denni\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\denni\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Project modules imported successfully.


## 1. Define the stock universe and download historical data

In [2]:
tickers = DEFAULT_TICKERS
start_date = '2021-01-01'
end_date = None

prices = download_price_data(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)

prices.tail()

Ticker,AAPL,AMZN,GOOG,JPM,META,MSFT,NVDA,TSLA,UNH,XOM
Date,,,,,,,,,,
2026-03-31,253.789993,208.270004,286.859985,292.662231,572.130005,370.170013,174.399994,371.750000,270.589996,169.660004
2026-04-01,255.630005,210.570007,294.899994,293.876038,579.229980,369.369995,175.750000,381.260010,273.980011,160.779999
2026-04-02,255.919998,209.770004,294.459991,293.100006,574.460022,373.459991,177.389999,360.589996,277.260010,160.690002
2026-04-06,258.859985,212.789993,297.660004,295.450012,573.020020,372.880005,177.639999,352.820007,281.359985,163.369995
2026-04-07,253.500000,213.770004,303.929993,297.399994,575.049988,372.290009,178.100006,346.649994,307.730011,163.910004


## 2. Explore historical price performance

In [3]:
plot_price_history(prices)

c:\Users\denni\anaconda3\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()


## 3. Cluster stocks by risk using K-Means

We summarize each stock using annualized return, annualized volatility, Sharpe ratio, and downside volatility, then assign a risk label based on cluster volatility.

In [4]:
clusters = cluster_stocks(returns)
clusters

,annual_return,annual_vol,sharpe,downside_vol,cluster,risk_group
Ticker,,,,,,
NVDA,0.630839,0.515451,1.223858,0.284624,0,High
XOM,0.339254,0.271330,1.250338,0.158194,2,Low
GOOG,0.289078,0.307287,0.940741,0.180005,2,Low
JPM,0.220533,0.243344,0.906260,0.146288,2,Low
AAPL,0.172013,0.277122,0.620709,0.162985,2,Low
MSFT,0.144902,0.260981,0.555219,0.156904,2,Low
TSLA,0.245254,0.598127,0.410037,0.343258,1,Medium
META,0.241350,0.432861,0.557568,0.266465,1,Medium
AMZN,0.117030,0.349148,0.335186,0.209102,1,Medium


In [5]:
plot_cluster_map(clusters)

c:\Users\denni\anaconda3\lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



## 4. Build technical indicators for machine learning

In [6]:
feature_panel = {ticker: build_technical_features(prices[ticker]) for ticker in prices.columns}
dataset = prepare_ml_dataset(feature_panel)

dataset.head()

,Date,close,return_1d,return_5d,return_10d,volatility_10d,volatility_21d,sma_10_ratio,sma_21_ratio,sma_50_ratio,rsi_14,macd,macd_signal,bb_width,volume_change_5d,target_next_return,target_up,ticker
49,2021-03-16,122.304642,0.012743,0.036997,0.003597,0.024773,0.023877,0.034699,0.010500,-0.032398,50.950722,-2.313946,-2.784969,0.123683,0.0,-0.006451,0,AAPL
50,2021-03-17,121.515694,-0.006451,0.039840,0.022120,0.023361,0.023711,0.025742,0.007235,-0.037979,49.560220,-2.009807,-2.629937,0.112244,0.0,-0.033905,0,AAPL
51,2021-03-18,117.395683,-0.033905,-0.011725,0.003330,0.025531,0.024471,-0.009361,-0.023043,-0.069120,42.965771,-2.077280,-2.519405,0.104019,0.0,-0.004480,0,AAPL
52,2021-03-19,116.869728,-0.004480,-0.008593,-0.011777,0.025314,0.024446,-0.012639,-0.023757,-0.072370,42.193926,-2.148427,-2.445210,0.092508,0.0,0.028336,1,AAPL
53,2021-03-22,120.181320,0.028336,-0.004839,0.060416,0.022283,0.025400,0.009499,0.006432,-0.045002,48.470677,-1.915513,-2.339270,0.089058,0.0,-0.006889,0,AAPL


## 5. Train Random Forest models

We train:
- a **regressor** for next-day return
- a **classifier** for next-day up/down direction


In [7]:
regressor, classifier = train_random_forest_models(dataset)

latest_rows = []
for ticker, df in feature_panel.items():
    row = df.tail(1).copy()
    row['ticker'] = ticker
    latest_rows.append(row)
latest_feature_rows = __import__('pandas').concat(latest_rows)

predictions = predict_next_day_scores(latest_feature_rows, regressor, classifier)
predictions

,ticker,predicted_return,prob_up,ai_score
Date,,,,
2026-04-07,TSLA,0.004473,0.551656,0.775828
2026-04-07,MSFT,0.001059,0.570653,0.735326
2026-04-07,JPM,0.000768,0.533698,0.666849
2026-04-07,XOM,0.000757,0.534891,0.617446
2026-04-07,AMZN,0.000695,0.538945,0.569472
2026-04-07,NVDA,0.000676,0.519185,0.509592
2026-04-07,AAPL,0.000630,0.514151,0.457075
2026-04-07,META,-0.000288,0.479721,0.389861
2026-04-07,UNH,-0.002057,0.490372,0.345186


In [8]:
plot_feature_importance(regressor, TECHNICAL_FEATURES)

## 6. Optimize a portfolio with PyPortfolioOpt

We create two optimized portfolios:
- **AI-Optimized**: blends historical expected returns with ML-predicted alpha
- **Classical Mean-Variance**: uses historical expected returns only


In [9]:
risk_target = RISK_LEVEL_MAP['Balanced']
ai_alpha = predictions.set_index('ticker')['predicted_return'] * 252
selected = predictions.query('prob_up >= 0.5')['ticker'].tolist()
if len(selected) < 3:
    selected = predictions.head(6)['ticker'].tolist()

ai_weights, ai_perf = optimize_portfolio(prices[selected], risk_target=risk_target, ai_alpha=ai_alpha)
mv_weights, mv_perf = optimize_portfolio(prices, risk_target=risk_target, ai_alpha=None)

ai_perf, mv_perf

({'expected_return': 0.30809693750744727,
  'volatility': 0.22051562041553888,
  'sharpe': 1.3064695234040944},
 {'expected_return': 0.3428789496522958,
  'volatility': 0.259947556215534,
  'sharpe': 1.2420926526602258})

In [10]:
plot_weights(ai_weights, title='AI-Optimized Portfolio Weights')

## 7. Backtest strategies over time

The backtest compares:
- AI-Optimized
- MeanVariance
- EqualWeight


In [ ]:
backtest_returns, weights_history = run_backtest(prices, risk_target=risk_target)
performance_summary = summarize_performance(backtest_returns)

performance_summary

In [ ]:
plot_cumulative_returns(backtest_returns)

## 8. Interpretation

You can use this project in a portfolio to demonstrate:

- financial data collection and preprocessing
- feature engineering for time series
- unsupervised learning with K-Means
- supervised learning with Random Forest
- portfolio optimization with PyPortfolioOpt
- backtesting and strategy evaluation
- dashboard deployment with Streamlit


## Suggested Resume / Portfolio Description

**AI-Driven Portfolio Optimizer** — Built an end-to-end portfolio analytics app using `yfinance`, `scikit-learn`, `PyPortfolioOpt`, `Plotly`, and `Streamlit`. Engineered technical indicators, clustered stocks by risk using K-Means, predicted next-day returns with Random Forest, optimized allocations with mean-variance optimization, and backtested AI-assisted strategies over a 5-year horizon.